# Day 60 — Random Forests and Ensemble Methods

This notebook covers Ensemble Learning, Bootstrap Aggregating (Bagging), Random Feature Subspace Selection, Wisdom of the Crowd, and Scikit-Learn Random Forest Classification

## Validation goals
- Understand ensemble learning principles and how combining weak learners creates superior aggregate predictions
- Analyze the two primary sources of randomness in Random Forests: row-level bagging and column-level random feature selection
- Explore why random forests dramatically reduce variance and resist individual tree overfitting
- Implement a random forest classifier in Python using Scikit-Learn, evaluating test set accuracy and feature importances

### 1. What is a Random Forest?

### Concept Explanation

A Random Forest is an ensemble learning method used for both classification and regression. Instead of relying on a single decision tree, an ensemble model trains a "forest" of many distinct decision trees and aggregates their predictions.

**The Wisdom of the Crowd Analogy**

Imagine you need to estimate the weight of an ox at a country fair:
- If you ask a single random person on the street (equivalent to a single decision tree), their guess might be wildly inaccurate because they could focus on a weird quirk (e.g., "its horns look funny, so it must weigh 2,000 lbs").
- But if you poll 500 different fairgoers and average out all their guesses (equivalent to a Random Forest), the individual errors cancel each other out, and the collective estimate becomes remarkably accurate. This is the wisdom of the crowd.

### 2. How Random Forests Avoid Overfitting (The Two Secrets)

To ensure the trees in the forest don't all make the exact same mistakes, a Random Forest introduces two layers of randomness during training:

- Bagging (Bootstrap Aggregating): Each tree in the forest is trained on a slightly different random subset of your training data (sampled with replacement). Because no tree sees the exact same dataset, each learns a slightly different perspective.
- Random Feature Selection: When a tree is building a split at a node, it is only allowed to look at a random subset of features (e.g., choosing 3 out of 10 columns). This forces the trees to be decorrelated and prevents one dominant feature from hijacking every single tree in the forest.

At prediction time, the forest combines the output:
- Classification: Uses majority voting (e.g., if 85 out of 100 trees vote "Spam", the forest outputs "Spam").
- Regression: Takes the average of all the individual tree predictions.

### 3. Implementing a Random Forest in Python (Scikit-Learn)

Let's write a clean Python script using Scikit-learn to train a RandomForestClassifier on a classification task.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Setup synthetic dataset: Customer metrics vs. Churn (0 = Stayed, 1 = Churned)
np.random.seed(42)
X_data = np.random.rand(500, 4)  # 500 samples, 4 features (e.g., usage, age, support tickets, bill)
y_data = np.random.randint(0, 2, size=500)  # Binary labels

# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.20, random_state=42)

# 2. Instantiate the Random Forest Classifier
# n_estimators sets the number of decision trees in the forest
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 3. Train (fit) the model on our training data
model.fit(X_train, y_train)

# 4. Evaluate performance on the test set
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Random Forest successfully trained with {model.n_estimators} trees.")
print(f"Test Set Accuracy: {accuracy * 100:.2f}%\n")

# 5. Check Feature Importances
# Random forests automatically track which features were most useful across all trees
for idx, importance in enumerate(model.feature_importances_):
    print(f"Feature {idx + 1} Importance: {importance:.4f}")

Random Forest successfully trained with 100 trees.
Test Set Accuracy: 52.00%

Feature 1 Importance: 0.2640
Feature 2 Importance: 0.2501
Feature 3 Importance: 0.2605
Feature 4 Importance: 0.2253


### Guided Question:

What is an ensemble method, and why is a Random Forest generally much more robust and accurate than a single standalone Decision Tree?

### Answer:

**What is an Ensemble Method?**

An ensemble method is a machine learning technique that combines the predictions of multiple individual models—known as base learners (typically "weak learners," such as shallow decision trees)—to produce a single, superior aggregate prediction.

**The core intuition relies on the "wisdom of the crowds":** while individual models may make errors, suffer from high variance, or capture different blind spots, combining their outputs through voting (for classification) or averaging (for regression) dramatically cancels out individual mistakes and stabilizes overall performance.

**What is a Random Forest?**

A Random Forest is a specific type of ensemble method built entirely out of decision trees. It achieves its power by combining two key randomization techniques:
- Bootstrap Aggregating (Bagging): Each tree in the forest is trained on a distinct subset of the original training data created by sampling rows with replacement. As a result, every tree sees a slightly different variation of the dataset.Random Feature
- Subspace (Feature Randomness): At every node split within every tree, the algorithm is restricted to evaluating only a random subset of features (typically $\sqrt{p}$ out of $p$ total features) rather than all available columns.

**Why a Random Forest Outperforms a Standalone Decision Tree**

While a single standalone decision tree is prone to high variance, extreme sensitivity to training data fluctuations, and severe overfitting, a Random Forest systematically neutralizes these flaws through several mechanical advantages:

**1. Massive Reduction in Variance**

A single unconstrained decision tree has extremely high variance—change a few rows in the training set, and the entire tree structure reorganizes.
- By training dozens or hundreds of trees on independent bootstrap samples and averaging their predictions, the Random Forest smooths out erratic predictions.
- Mathematically, averaging $N$ decorrelated models reduces the variance of the ensemble by a factor proportional to $N$, yielding a stable, robust predictor.

**2. Tree Decorrelation via Feature Subsampling**

If a dataset contains one overwhelmingly dominant feature (e.g., a feature with high predictive power), every single tree in a naive bagging ensemble will pick that exact feature for its root node, making all the trees look nearly identical (highly correlated).
- By forcing each tree to choose splits from a random subset of features, Random Forests break this correlation. Trees are forced to discover secondary and tertiary patterns using alternative features.
- When diverse, decorrelated trees vote together, the collective accuracy climbs significantly.

**3. Immunity to Individual Overfitting**

An unconstrained standalone tree greedily carves up the feature space into microscopic, jagged boxes to achieve 0% training error, capturing noise and outliers along the way.
- While individual trees in a forest do overfit their specific bootstrap samples, they overfit in different, uncorrelated ways.
- The noise and spurious rules learned by Tree A are contradicted by Trees B through Z. When the forest aggregates the votes, the idiosyncratic noise cancels out, leaving behind only the true underlying signal.

**4. Smoother Decision Boundaries**
A single decision tree creates rigid, orthogonal, step-function decision boundaries that struggle with diagonal or continuous relationships. By averaging the outputs of hundreds of trees making slightly different axis-aligned splits, a Random Forest constructs a smooth, composite, pseudo-continuous decision boundary that generalizes far better to unseen test data.

### Guided Question:

What are the two primary ways a Random Forest introduces randomness during training to ensure its individual trees remain diverse?

### Answer:

The two primary ways a Random Forest introduces randomness during training—collectively known as ensemble randomization—are designed to break the correlation between individual trees, ensuring they make different types of errors that cancel each other out during aggregation.

**1. Bootstrap Aggregating (Row-Level Randomness / Bagging)**

The first layer of randomness occurs before a single tree is even built. Instead of training every tree on the exact same dataset, each tree is fed a unique bootstrap sample.
- How it works: The algorithm draws rows from the original training dataset with replacement, creating a new subset of the same size as the original. Because sampling is done with replacement, about 36.8% of the original data points are left out entirely (known as Out-of-Bag or OOB samples), while some rows are duplicated multiple times.
- The Effect: Because each tree sees a slightly different variation of the training distribution, each model develops unique internal biases and captures distinct nuances of the data.

**2. Random Feature Subspace Selection (Column-Level Randomness)**

While bagging varies the rows each tree sees, feature subsampling varies the columns available at every single decision point.
- How it works: At every node split within every tree, the algorithm is restricted to evaluating only a random subset of features (typically $\sqrt{p}$ for classification tasks or $p/3$ for regression, where $p$ is the total number of features) rather than inspecting all available columns. A new random subset of features is drawn at every node split.
- The Effect: This prevents a single overwhelmingly dominant or highly predictive feature from hijacking every tree in the forest. If a dataset has one master feature, a standard bagged tree would make that feature the root node every time, causing all trees to look nearly identical (high correlation). By forcing feature randomness, trees are compelled to discover secondary, tertiary, and non-linear patterns using alternative attributes.

### Guided Question:

In the Scikit-learn code snippet above, what does the n_estimators=100 parameter control, and how does changing that number impact the model?

### Answer:

In the provided scikit-learn code snippet, the n_estimators=100 parameter controls the total number of individual decision trees built and combined inside the Random Forest ensemble.

Setting it to 100 means the algorithm will bootstrap the training data 100 times, train 100 independent decision trees using random feature subspaces at each split, and then aggregate their final predictions (via majority voting for classification or averaging for regression).

**How Changing n_estimators Impacts the Model**
Modifying this parameter affects training time, memory consumption, and statistical stability, but in distinct ways compared to hyperparameters like max_depth.

**1. Increasing n_estimators (e.g., 500 or 1,000)**

- Greater Stability & Lower Variance: As you add more trees, the aggregate prediction becomes smoother and more robust. The law of large numbers takes effect, ensuring that idiosyncratic errors or noise captured by individual trees are effectively averaged out
- More Stable Feature Importances: The `model.feature_importances_ values` (computed as the mean impurity decrease across all trees) become much more reliable and stable because they are averaged over a larger statistical sample.
- Diminishing Returns: Performance gains plateau rapidly. Going from 10 trees to 100 trees yields a massive accuracy boost; going from 100 trees to 1,000 trees usually yields negligible accuracy improvement while multiplying your training time.
- Resource Cost: Training time and memory usage scale linearly with the number of trees ($O(N)$). Inference latency also increases slightly because every test sample must be evaluated by every tree in the forest.

**2. Decreasing n_estimators (e.g., 5 or 10)**

High Variance & Unstable Predictions: A small forest is prone to high variance. If you re-run training with a different random seed, your predictions and test accuracy can fluctuate wildly because a small sample of trees fails to fully represent the ensemble distribution.
- Noisy Feature Importances: Feature importance scores will vary significantly across runs because a few dominant trees can skew the global average.
- Fast Training: Ideal for rapid prototyping, hyperparameter tuning sweeps, or working with massive datasets where training a full forest is computationally expensive.

**A Crucial Distinction: Does High n_estimators Cause Overfitting?**

Unlike `max_depth` (where setting it too high or to `None` causes individual trees to memorize training data and overfit), increasing n_estimators does not cause overfitting.

Because of the mechanics of bagging (Bootstrap Aggregating), adding more trees to a random forest continuously reduces variance without increasing model bias. Once you pass a certain threshold (often around 100 to 300 trees depending on the dataset), test error flattens out completely rather than increasing.